# 02 — Cross-Asset Financial Market EDA

## S&P 500 × Gold × Oil × DXY × VIX

### سؤال اصلی
**آیا تغییرات بازار سهام آمریکا با طلا، نفت، شاخص دلار و نوسان‌پذیری بازار رابطه آماری دارد؟**

این Notebook بعد از آماده‌سازی S&P اجرا می‌شود و مراحل `Merge → Transform → Scale → Distribution → Test → Correlation → Regression → Rolling Analysis → Interpretation` را انجام می‌دهد.

## نکته روش‌شناسی مهم
- برای تحلیل اصلی، **S&P 500 رسمی (`^GSPC`)** از منبع بازار دریافت می‌شود.
- فایل ارسالی S&P در Notebook 01 برای extraction/cleaning و ساخت proxy استفاده می‌شود.
- Gold=`GC=F`، WTI=`CL=F`، DXY=`DX-Y.NYB` و VIX=`^VIX` هستند.
- VIX یک شاخص نوسان است؛ بنابراین علاوه بر `VIX_Return`، سطح VIX نیز برای تفسیر ریسک بررسی می‌شود.
- همبستگی و رگرسیون **رابطه آماری هستند، نه علیت**.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE=42

proxy_path=Path("../data/sp500_market_proxy.csv")
if not proxy_path.exists(): proxy_path=Path("data/sp500_market_proxy.csv")
proxy=pd.read_csv(proxy_path,parse_dates=["Date"])
print(proxy.shape, proxy.Date.min().date(), proxy.Date.max().date())

## 1) دریافت داده‌های cross-asset
داده‌های بیرونی هنگام اجرای Notebook دریافت می‌شوند. اگر محیط اجرا اینترنت نداشته باشد، این سلول عمداً خطای واضح می‌دهد و نباید با داده ساختگی جایگزین شود.

In [ ]:
# Install once if needed:
# %pip install yfinance
import yfinance as yf

SYMBOLS={"SP500":"^GSPC","Gold":"GC=F","Oil_WTI":"CL=F","DXY":"DX-Y.NYB","VIX":"^VIX"}
start=proxy.Date.min().strftime("%Y-%m-%d")
end=(proxy.Date.max()+pd.Timedelta(days=1)).strftime("%Y-%m-%d")

raw_ext=yf.download(list(SYMBOLS.values()),start=start,end=end,interval="1d",auto_adjust=True,progress=False,threads=False)
if raw_ext.empty: raise RuntimeError("External market data could not be downloaded. Check internet access and rerun.")

if isinstance(raw_ext.columns,pd.MultiIndex):
    close=raw_ext["Close"].copy()
else:
    close=raw_ext[["Close"]].copy()
close=close.rename(columns={v:k for k,v in SYMBOLS.items()})
close.index=pd.to_datetime(close.index).tz_localize(None)
close=close.reset_index().rename(columns={close.index.name or "index":"Date"})
print(close.shape)
display(close.head())

## 2) کنترل کیفیت داده‌های خارجی و ادغام

In [ ]:
assert close.Date.notna().all()
assert close.Date.duplicated().sum()==0
print("Missing by series (%):")
display((close.isna().mean()*100).round(2).to_frame("missing_%"))

df=proxy[["Date","SP500_Proxy_Return","SP500_Proxy_Index"]].merge(close,on="Date",how="inner")
for c in ["SP500","Gold","Oil_WTI","DXY"]:
    df[f"{c}_Return"]=df[c].pct_change(fill_method=None)
df["VIX_Return"]=df["VIX"].pct_change(fill_method=None)

needed=["SP500_Return","Gold_Return","Oil_WTI_Return","DXY_Return","VIX_Return"]
df=df.dropna(subset=needed).reset_index(drop=True)
print("Merged shape:",df.shape)
print("Analysis period:",df.Date.min().date(),"to",df.Date.max().date())
display(df.head())

## 3) مقایسه S&P رسمی با proxy فایل ارسالی
این کنترل نشان می‌دهد proxy ساخته‌شده از constituentها تا چه حد رفتار بازار رسمی را دنبال می‌کند.

In [ ]:
proxy_check=df[["SP500_Return","SP500_Proxy_Return"]].corr().iloc[0,1]
print(f"Correlation between official S&P return and constituent proxy return: {proxy_check:.3f}")
plt.figure(figsize=(8,4))
plt.plot(df.Date,df.SP500_Return,label="Official S&P 500")
plt.plot(df.Date,df.SP500_Proxy_Return,label="Constituent proxy",alpha=.7)
plt.legend(); plt.title("Official S&P 500 vs Constituent Proxy")
plt.show()

## 4) آمار توصیفی بازده‌ها

In [ ]:
return_cols=["SP500_Return","Gold_Return","Oil_WTI_Return","DXY_Return","VIX_Return"]
desc=df[return_cols].describe().T
desc["skewness"]=df[return_cols].skew()
desc["excess_kurtosis"]=df[return_cols].kurtosis()
display(desc.round(4))

## 5) توزیع، Boxplot و Outlier

In [ ]:
for c in return_cols:
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    sns.histplot(df[c],bins=80,kde=True,ax=ax[0])
    ax[0].set_title(f"Distribution — {c}")
    sns.boxplot(x=df[c],ax=ax[1])
    ax[1].set_title(f"Boxplot — {c}")
    plt.tight_layout(); plt.show()

out=[]
for c in return_cols:
    q1,q3=df[c].quantile([.25,.75]); iqr=q3-q1
    lo,hi=q1-1.5*iqr,q3+1.5*iqr
    n=int(((df[c]<lo)|(df[c]>hi)).sum())
    out.append([c,q1,q3,iqr,lo,hi,n,n/len(df)*100])
out_df=pd.DataFrame(out,columns=["variable","Q1","Q3","IQR","lower","upper","outlier_count","outlier_pct"])
display(out_df.round(4))

## 6) آزمون نرمال بودن + Q-Q Plot
H0: داده از توزیع نرمال می‌آید. اگر `p < 0.05` باشد، H0 رد می‌شود. نتیجه باید همراه skewness، kurtosis و Q-Q plot تفسیر شود.

In [ ]:
normality=[]
for c in return_cols:
    x=df[c].dropna()
    sample=x.sample(min(5000,len(x)),random_state=RANDOM_STATE)
    W,p=stats.shapiro(sample)
    normality.append([c,W,p,"Reject H0" if p<.05 else "Fail to reject H0"])
    plt.figure(figsize=(5,5)); stats.probplot(x,dist="norm",plot=plt); plt.title(f"Q-Q Plot — {c}"); plt.show()
display(pd.DataFrame(normality,columns=["variable","W","p_value","conclusion"]).round(6))

## 7) Scaling
Scaling در این پروژه برای نشان دادن مهارت Data Analyst/ML preprocessing است؛ تفسیر اقتصادی بازده‌ها قبل از scaling انجام می‌شود.

In [ ]:
std=StandardScaler().fit_transform(df[return_cols])
mm=MinMaxScaler().fit_transform(df[return_cols])
scaled=pd.concat([pd.DataFrame(std,columns=[c+"_z" for c in return_cols]),pd.DataFrame(mm,columns=[c+"_minmax" for c in return_cols])],axis=1)
display(scaled.head())

## 8) شاخص پایه 100 برای مقایسه روند قیمت‌ها

In [ ]:
price_cols=["SP500","Gold","Oil_WTI","DXY","VIX"]
indexed=df[["Date"]+price_cols].copy()
for c in price_cols: indexed[c]=indexed[c]/indexed[c].iloc[0]*100
plt.figure(figsize=(14,6))
for c in price_cols: plt.plot(indexed.Date,indexed[c],label=c)
plt.legend(); plt.title("Cross-Asset Indexed Levels (Start = 100)"); plt.ylabel("Index"); plt.show()

## 9) Pearson و Spearman Correlation

In [ ]:
pearson=df[return_cols].corr("pearson")
spearman=df[return_cols].corr("spearman")
for name,mat in [("Pearson",pearson),("Spearman",spearman)]:
    plt.figure(figsize=(9,7)); sns.heatmap(mat,annot=True,fmt=".2f",center=0,cmap="coolwarm"); plt.title(f"{name} Correlation — Daily Returns"); plt.show()
display(pearson.round(3))

## 10) Scatter Plot و Pair Plot

In [ ]:
pairs=[("SP500_Return","Gold_Return"),("SP500_Return","Oil_WTI_Return"),("SP500_Return","DXY_Return"),("SP500_Return","VIX_Return")]
for x,y in pairs:
    plt.figure(figsize=(6,4)); sns.regplot(data=df,x=x,y=y,scatter_kws={"alpha":.25},line_kws={"linewidth":2}); plt.title(f"{x} vs {y}"); plt.show()

pair_sample=df[return_cols].sample(min(1200,len(df)),random_state=RANDOM_STATE)
sns.pairplot(pair_sample,corner=True,plot_kws={"alpha":.25}); plt.show()

## 11) آزمون معناداری همبستگی و رگرسیون ساده

In [ ]:
tests=[]; regs=[]
y=df["SP500_Return"]
for _,xcol in pairs:
    x=df[xcol]
    r,p=stats.pearsonr(x,y)
    model=LinearRegression().fit(x.to_numpy().reshape(-1,1),y)
    pred=model.predict(x.to_numpy().reshape(-1,1))
    tests.append([xcol,r,p,"Significant" if p<.05 else "Not significant"])
    regs.append([xcol,float(model.coef_[0]),float(model.intercept_),float(r2_score(y,pred))])

corr_tests=pd.DataFrame(tests,columns=["predictor","r","p_value","conclusion"])
reg_df=pd.DataFrame(regs,columns=["predictor","slope","intercept","R2"])
display(corr_tests.round(5)); display(reg_df.round(6))

## 12) Rolling Correlation
برای دیدن تغییر رابطه در طول زمان از پنجره 60 روزه استفاده می‌کنیم.

In [ ]:
for _,xcol in pairs:
    roll=df.SP500_Return.rolling(60).corr(df[xcol])
    plt.figure(figsize=(13,4)); plt.plot(df.Date,roll); plt.axhline(0,linewidth=1); plt.title(f"60-Day Rolling Correlation — S&P vs {xcol}"); plt.ylabel("Correlation"); plt.show()

## 13) Rolling Volatility و رابطه VIX با ریسک بازار

In [ ]:
df["SP500_Vol21"]=df.SP500_Return.rolling(21).std()*np.sqrt(252)
plt.figure(figsize=(13,5)); plt.plot(df.Date,df.SP500_Vol21,label="S&P 500 21D annualized volatility"); plt.plot(df.Date,df.VIX/100,label="VIX / 100",alpha=.8); plt.legend(); plt.title("Realized Volatility vs VIX Level"); plt.show()
print("Correlation: realized volatility vs VIX level =",df[["SP500_Vol21","VIX"]].corr().iloc[0,1].round(3))

# نتیجه‌گیری نهایی
این بخش هنگام اجرا اعداد واقعی را از داده‌های دریافت‌شده تولید می‌کند. در گزارش نهایی باید به جای ادعای علیت، از عبارت‌هایی مثل «همبستگی»، «رابطه آماری» و «شواهد نمونه» استفاده شود.

### نکات قابل دفاع در ارائه
1. داده خام S&P ابتدا ساختارشناسایی و فقط بلوک Close برای بازده انتخاب شد.
2. Missing Valueها به‌صورت شفاف گزارش شدند و قیمت جعلی تولید نشد.
3. outlierها حذف نشدند چون در بازار مالی می‌توانند رویداد واقعی باشند.
4. بازده روزانه برای مقایسه دارایی‌ها مناسب‌تر از قیمت خام است.
5. Pearson و Spearman هر دو بررسی شدند.
6. معناداری آماری جدا از بزرگی اثر تفسیر شد.
7. Rolling correlation نشان می‌دهد روابط در زمان ثابت نیستند.
8. VIX به عنوان شاخص ریسک/انتظارات نوسان در کنار بازده و نوسان تحقق‌یافته بررسی شد.
9. Scaling صرفاً یک مرحله preprocessing است و معنای اقتصادی داده را عوض نمی‌کند.

## منابع و بازتولیدپذیری
- S&P constituent panel: `SnP_daily_update.csv` ارسالی پروژه.
- Gold / WTI / DXY / VIX / official S&P 500: Yahoo Finance via `yfinance`.
- برای اجرای دقیق cross-asset، اینترنت فعال و نصب `requirements.txt` لازم است.

### ساختار پیشنهادی GitHub
```text
Cross_Asset_EDA_Final/
├── notebooks/
│   ├── 01_Prepare_SP500.ipynb
│   └── 02_Cross_Asset_EDA.ipynb
├── data/
│   ├── sp500_market_proxy.csv
│   ├── sp500_quality_report.csv
│   └── SnP_daily_update.csv   # optional / Git LFS
├── README.md
├── requirements.txt
└── .gitignore
```